# 🧬 Phase 1: Biomedical KG — Data Download & RAPIDS GPU Preprocessing

**Project:** BioKG Disease Link Predictor  
**Dataset:** DRKG (Drug Repurposing Knowledge Graph) — 5.8M biological triples  
**Tech:** NVIDIA RAPIDS cuDF/cuGraph + BioBridge Alignment  

---

## 🎓 Phase 1 Objective
1. **GPU Speedup**: Benchmark RAPIDS vs pandas (aiming for 10x+ speedup)
2. **Data Cleansing**: Smart extraction of 5.8M triples
3. **BioBridge Alignment**: Understanding node types (Protein, Drug, Disease) to prepare for BioBridge embeddings

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ Stop! No GPU detected."
print(f"\n✅ GPU Connected: {torch.cuda.get_device_name(0)}")

## Step 1: Install NVIDIA RAPIDS
This takes ~3-5 minutes. ☕

In [ ]:
import torch
cuda_version = torch.version.cuda
major = int(cuda_version.split('.')[0]) if cuda_version else 12
rapids_suffix = 'cu11' if major == 11 else 'cu12'

print(f"📦 Installing RAPIDS for CUDA {cuda_version}...")
!pip install cudf-{rapids_suffix} cugraph-{rapids_suffix} --extra-index-url=https://pypi.nvidia.com -q
print("✅ RAPIDS installed.")

## Step 2: Download & Smart Extraction

In [ ]:
import requests
import tarfile
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('data/raw')
if DATA_DIR.exists():
    import shutil
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("📥 Downloading DRKG dataset...")
url = 'https://dgl-data.s3-accelerate.amazonaws.com/dataset/DRKG/drkg.tar.gz'
dest = DATA_DIR / 'drkg.tar.gz'

response = requests.get(url, stream=True)
total = int(response.headers.get('content-length', 0))
with open(dest, 'wb') as f, tqdm(total=total, unit='iB', unit_scale=True) as pb:
    for chunk in response.iter_content(8192):
        pb.update(f.write(chunk))

print("\n📦 Extracting...")
with tarfile.open(dest, 'r:gz') as tar:
    tar.extractall(path=DATA_DIR)

# Locate the largest TSV file
tsv_files = list(DATA_DIR.rglob('*.tsv'))
DRKG_TSV = max(tsv_files, key=lambda f: f.stat().st_size)

print(f"\n✅ Target File Found: {DRKG_TSV.name} ({DRKG_TSV.stat().st_size / 1024**2:.1f} MB)")

## Step 3: GPU vs CPU Benchmark

> **Fix:** `cudf.read_csv` does not take `encoding`. It handles raw bytes by default. We keep `latin-1` for pandas only.

In [ ]:
import time
import cudf
import pandas as pd

print('⚡ Starting Benchmark...')

print('\n🐢 Loading with pandas (CPU)...')
t0 = time.time()
pdf = pd.read_csv(str(DRKG_TSV), sep='\t', header=None, 
                  names=['head', 'relation', 'tail'], encoding='latin-1')
cpu_time = time.time() - t0
print(f'   Time: {cpu_time:.2f}s | Rows: {len(pdf):,}')

print('\n⚡ Loading with cuDF (GPU)...')
t0 = time.time()
gdf = cudf.read_csv(str(DRKG_TSV), sep='\t', header=None, 
                   names=['head', 'relation', 'tail']) 
gpu_time = time.time() - t0
print(f'   Time: {gpu_time:.2f}s | Rows: {len(gdf):,}')

speedup = cpu_time / max(gpu_time, 0.001)
print(f'\n🚀 cuDF Speedup: {speedup:.1f}x!')

print('\n📊 Data Preview:')
print(gdf.head().to_pandas().to_string(index=False))

## Step 4: BioBridge Node Exploration

Like the **BioBridge-PrimeKG** pattern, let's categorize our nodes by modality.

In [ ]:
print("🔍 Analyzing Node Modalities (BioBridge Alignment)...")

# Extract types on GPU
head_parts = gdf['head'].str.split('::')
gdf['head_type'] = head_parts.list.get(0)

type_counts = gdf['head_type'].value_counts().to_pandas()

print("\n🧬 Top BioBridge Modalities found in DRKG:")
relevant_types = ['Gene', 'Compound', 'Disease']
for t in relevant_types:
    count = type_counts.filter(like=t).sum()
    print(f"   • {t:<10}: {int(count):>10,}")

## Step 5: Save To Drive (Parquet)

We save as **Parquet** (standard MLOps format) directly to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = Path('/content/drive/MyDrive/BioKG_Project/')
SAVE_PATH.mkdir(parents=True, exist_ok=True)

print("💾 Archiving KG to Google Drive as binary Parquet...")
gdf.to_parquet(SAVE_PATH / 'drkg.parquet')
print(f"✅ Done! Persistent data at: {SAVE_PATH}/drkg.parquet")